In [ ]:
import retrieval_pipeline as rag

In [ ]:
import subprocess
import sys

before = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]

result = subprocess.run(
    [sys.executable, "../rag_deploy/indexing_pipeline.py"],
    cwd="../rag_deploy",
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

after = rag.pinecone_index_dense.describe_index_stats()["total_vector_count"]
print(f"dense index vector count: {before} before this run -> {after} after (unchanged = idempotent)")

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field


class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, description="The user's question.")
    top_k: Optional[int] = Field(
        default=None, ge=1, le=10,
        description="Number of cited chunks to return. Defaults to the retriever's configured top_k.",
    )

class Source(BaseModel):
    chunk_id: str
    title: str
    text: str
    score: float


class QueryResponse(BaseModel):
    answer: str
    sources: List[Source]


# A request with no "question" key, or question="", now fails validation automatically:
try:
    QueryRequest(question="")
except Exception as e:
    print("Rejected, as expected:", e)

In [ ]:
from pathlib import Path

print(Path("../rag_deploy/main.py").read_text(encoding="utf-8"))

In [ ]:
import subprocess
import sys
import time

import requests

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--port", "8000"],
    cwd="../rag_deploy",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

server_up = False
for _ in range(20):
    # If the process already died, stop waiting and go straight to the error path
    if server.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).ok:
            server_up = True
            break
    except requests.ConnectionError:
        time.sleep(0.5)

if not server_up:
    server.terminate()
    server.wait()
    print("Server failed to start. Captured output:\n")
    print(server.stdout.read())
    raise SystemExit("Aborting — see uvicorn output above for the actual error.")

print("GET /health  ->", requests.get("http://127.0.0.1:8000/health").json())

resp = requests.post(
    "http://127.0.0.1:8000/query",
    json={"question": "Can a registered general nurse with a diploma apply for a nursing top-up at KNUST?"},
)
print("POST /query  ->", resp.status_code)
answer = resp.json()
print("answer:", answer["answer"])
print(f"cited {len(answer['sources'])} sources, top score {answer['sources'][0]['score']:.3f}")

print("\nSwagger UI (open while the server is running): http://127.0.0.1:8000/docs")

server.terminate()
server.wait()